In [ ]:
import src.utils
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import sklearn.linear_model
import pathlib
import os
import sklearn.linear_model
import cmocean
import pandas as pd
import copy
import scipy.stats
import matplotlib as mpl
import matplotlib.patheffects as pe

## RNG
rng = np.random.default_rng()

## color palette
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## paths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
# SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Load data

In [ ]:
def window(x):
    """window the data"""
    x_ = src.utils.get_windowed(x, window_size=480, stride=120)
    return x_.isel(year=slice(None, -1))

In [ ]:
forced, anom = src.utils.load_consolidated()
total = xr.merge(
    [
        forced["sst"] + anom["sst"],
        forced["sst_comp"],
    ]
)

## add T_3 data
Th = src.utils.load_cesm_indices()
T = xr.merge([(Th["T_3"] ** i).rename(f"T{i}") for i in range(3)])
total = xr.merge([total, T])

## window the data
total = window(total)
total = total.isel(year=slice(None, -1))

## get forced
forced_windowed = window(forced[["sst", "sst_comp"]])
forced_windowed = forced_windowed.groupby("time.month").mean()

## do regression

In [ ]:
def regress_pinv(X, x_vars, y_var, dims=["time", "member"]):
    """do nonlinear regression"""

    ## prep data
    stack = lambda x: x.stack(s=dims)  # .transpose(..., "s")
    X_ = stack(X[x_vars].to_dataarray(dim="v")).transpose(..., "v", "s")
    Y_ = stack(X[y_var]).transpose(..., "s")

    ## build coordinates for new array
    coords = {}
    for d in Y_.dims:
        if d != "s":
            coords[d] = Y_[d]

    ## add "v"
    coords["v"] = X_.v

    ## empty array to hold results
    m = xr.DataArray(coords=coords, dims=list(coords))

    ## do regression
    X_pinv = np.linalg.pinv(X_.values)
    m.values = np.einsum("...i,...ij->...j", Y_.values, X_pinv)

    return m.rename({"v": "param"}).squeeze(drop=True)


def regress_pinv_bymonth(x, **kwargs):
    """regress by month"""
    return x.groupby("time.month").map(regress_pinv, **kwargs)


def get_eq(x):
    return x.sel(longitude=slice(80, 280), latitude=slice(-5, 5)).mean("latitude")

In [ ]:
## compute coefficients
m = regress_pinv_bymonth(total, x_vars=["T0", "T1", "T2"], y_var="sst")

## reconstruct data
m = src.utils.reconstruct_fn(
    components=total["sst_comp"],
    scores=m,
    fn=get_eq,
)

## reconstruct forced
raw = src.utils.reconstruct_wrapper(
    forced_windowed,
    fn=get_eq,
)["sst"]

In [ ]:
sel = lambda x: x.sel(year=2010).mean("month")

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(m.longitude, sel(m.sel(param="T1")))
ax.plot(m.longitude, sel(m.sel(param="T2")))
ax.plot(m.longitude, sel(m.sel(param="T0")) - sel(raw))
ax.axhline(0, ls="--", c="k", lw=1)

plt.show()

In [ ]:
sel_mon = lambda x: x.mean("month")
delta = lambda x: sel_mon(x) - sel_mon(x).isel(year=0)
sel = lambda x: delta(x).T
# sel = lambda x : sel_(x)-sel_(x).mean("longitude")

## specify kwargs
kwargs = dict(
    extend="both",
    cmap="cmo.balance",
    # levels=src.utils.make_cb_range(.5,.05),
    levels=src.utils.make_cb_range(2.5, 0.25),
)

fig, axs = plt.subplots(1, 3, figsize=(8, 5))

axs[0].contourf(
    raw.longitude,
    raw.year,
    sel(raw),
    # sel(m)-sel(m).mean("longitude"),
    **kwargs,
)

axs[1].contourf(
    m.longitude,
    m.year,
    sel(m.sel(param="T0")),
    # sel(m)-sel(m).mean("longitude"),
    **kwargs,
)

axs[2].contourf(
    m.longitude,
    m.year,
    # sel(m.sel(param="T2")) * 1e1,
    10 * (sel(m.sel(param="T0")) - sel(raw)),
    **kwargs,
)

# for ax in axs:
#     ax.set_ylim([1875, 2010])

for ax in axs[1:]:
    ax.set_yticks([])

src.utils.add_vticks(axs, xticks=[80, 180, 280], xlines=[180])

plt.show()

In [ ]:
# plt.plot(total["T2"].std(["time"]).mean("member"))
get_Tw = lambda x: x.sel(longitude=slice(120, 180)).mean("longitude")
get_Te = lambda x: x.sel(longitude=slice(220, 280)).mean("longitude")
get_dTdx = lambda x: get_Tw(x) - get_Te(x)

## compute dTdx
# dTdx = src.utils.reconstruct_wrapper(total[["sst", "sst_comp"]], get_dTdx)
# dTdx = dTdx.rename({"sst": "dTdx"})

In [ ]:
delta = lambda x: x / x.isel(year=0) - 1
# delta = lambda x : x

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(raw.year, delta(get_dTdx(raw.mean("month"))))
ax.plot(m.year, delta(get_dTdx(m.mean("month")).sel(param="T0")))

ax_kwargs = dict(ls="--", c="k", lw=1)
ax.axhline(0, **ax_kwargs)
ax.axvline(1980, **ax_kwargs)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(m.longitude, m.sel(param="T2").mean("month").isel(year=0))
ax.axvline(180, **ax_kwargs)
ax.axhline(**ax_kwargs)
plt.show()

Load ERSST data

In [ ]:
## load ERSST
sst_ersst = xr.open_dataset(DATA_FP / "ersstv5.185401-202412.nc")
sst_ersst = sst_ersst.rename({"lat": "latitude", "lon": "longitude"})
sst_ersst = sst_ersst.assign_coords(
    {"time": xr.date_range(start="1854-01", end="2024-12", freq="MS")},
).sel(time=slice("1980-01", "2023-12"))

## get equatorial range
sst_ersst = get_eq(sst_ersst)
sst_ersst = sst_ersst

## update coords
T_3_ersst = sst_ersst["ssta"].sel(longitude=slice(210, 270)).mean("longitude")

## regression coeffs
T_ersst = xr.merge([(T_3_ersst**i).rename(f"T{i}") for i in range(3)])

## merge data
X_ersst = xr.merge([sst_ersst, T_ersst])

In [ ]:
## compute coefficients
m_ersst = regress_pinv_bymonth(
    X_ersst, x_vars=["T0", "T1", "T2"], y_var="sst", dims=["time"]
)

# ## reconstruct forced
# raw = src.utils.reconstruct_wrapper(
#     forced_windowed.mean("month"),
#     fn=get_eq,
# )["sst"]

Look at seasonal...because variance in winter is largest!!
This might be reason $T_0$ and $T_2$ don't match: different variance in different seasons?

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 3))
for ax, p in zip(axs, list(m.param)):
    ax.plot(m.longitude, m.sel(param=p).mean("month").isel(year=0), label="CESM2")
    ax.plot(m_ersst.longitude, m_ersst.mean("month").sel(param=p), label="ERSST")
    # ax.axhline(**ax_kwargs)
    ax.axvline(180, **ax_kwargs)

for ax in axs[1:]:
    ax.axhline(0, **ax_kwargs)
axs[0].legend()
plt.show()

In [ ]:
var_ersst = X_ersst["T2"].groupby("time.month").mean()
var_cesm = total["T2"].groupby("time.month").mean(["time", "member"])

Look at seasonality of rectification

In [ ]:
sel_mon = lambda x: x.mean("month")
delta = lambda x: sel_mon(x) - sel_mon(x).isel(year=0)
sel = lambda x: delta(x).T
# sel = lambda x : sel_(x)-sel_(x).mean("longitude")

## get rectification effect and difference
rect_ersst = m_ersst.sel(param="T2") * var_ersst
rect_cesm = m.sel(param="T2") * var_cesm
rect_diff = rect_ersst - rect_cesm.isel(year=0).interp({"longitude": m_ersst.longitude})

## specify kwargs
kwargs = dict(
    extend="both",
    cmap="cmo.balance",
    # levels=src.utils.make_cb_range(.5,.05),
    levels=src.utils.make_cb_range(0.3, 0.06),
)

fig, axs = plt.subplots(1, 3, figsize=(8, 3))

axs[0].contourf(
    rect_ersst.longitude,
    rect_ersst.month,
    rect_ersst,
    **kwargs,
)

axs[1].contourf(
    rect_cesm.longitude,
    rect_cesm.month,
    rect_cesm.isel(year=0).T,
    **kwargs,
)

axs[2].contourf(
    rect_diff.longitude,
    rect_diff.month,
    -rect_diff,
    **kwargs,
)

for ax in axs:
    ax.axvline(180, **ax_kwargs)
    ax.set_xticks([80, 180, 280])

for ax in axs[1:]:
    ax.set_yticks([])

plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3), layout="constrained")
axs[0].plot(rect_cesm.longitude, rect_cesm.isel(year=0).mean("month"))
axs[0].plot(rect_ersst.longitude, rect_ersst.mean("month"))
axs[1].plot(rect_ersst.longitude, rect_diff.mean("month"), c="k")
for ax in axs:
    ax.axhline(**ax_kwargs)
src.utils.add_vticks(axs, xticks=[80, 150, 180, 280], xlines=[150, 180])
src.utils.set_lims(axs)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3), layout="constrained")

axs[0].contourf(
    rect_cesm.longitude,
    rect_cesm.year,
    rect_cesm.mean("month").T,
    **kwargs,
)

axs[1].contourf(
    rect_cesm.longitude,
    rect_cesm.year,
    6 * delta(rect_cesm).T,
    **kwargs,
)

src.utils.add_vticks(axs, xticks=[150, 180, 280], xlines=[150, 180])
for ax in axs:
    ax.set_ylim([1870, 2010])

plt.show()

Idea: remove ENSO signal with least squares prediction...

In [ ]:
get_w = lambda x: x.sel(longitude=slice(110, 180)).mean("longitude")
get_w2 = lambda x: x.sel(longitude=slice(80, 150)).mean("longitude")
get_e = lambda x: x.sel(longitude=slice(180, 280)).mean("longitude")


def get_zg(x):
    """get zonal gradient"""

    return get_w(x) - get_e(x)


def get_zg2(x):
    """get zonal gradient"""

    return get_w2(x) - get_e(x)

In [ ]:
X_ersst_ann = X_ersst.groupby("time.year").mean()
X_ersst_recon = (
    X_ersst[["T0", "T1", "T2"]].to_dataarray(dim="param").groupby("time.month")
    * m_ersst
).sum("param")
X_ersst_ann_recon = X_ersst_recon.groupby("time.year").mean()
X_ersst_ann_filt = X_ersst_ann["sst"] - X_ersst_ann_recon
X_ersst_ann_filt = X_ersst_ann_filt.to_dataset(name="sst")

# .groupby("time.month")

note: filtered signal much less noisy!

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
# ax.plot(X_ersst_ann.year, get_zg(X_ersst_ann)["sst"])
ax.plot(X_ersst_ann.year, get_zg(X_ersst_ann_filt)["sst"])
# ax.plot(X_ersst_ann.year, get_e(X_ersst_ann_filt)["sst"])
# ax.plot(X_ersst_ann.year, get_w(X_ersst_ann_filt)["sst"])
# ax.plot(X_ersst_ann.year, get_w(X_ersst_ann)["sst"]-29.3)
# ax.plot(X_ersst_ann.year, get_zg2(X_ersst_ann)["sst"])
# ax.set_ylim([0,None])
plt.show()

In [ ]:
fn_ = lambda x: x.sel(longitude=slice(180, 280)).mean("longitude")
fn = lambda x: fn_(x) - fn_(x).mean()
plt.plot(get_e(X_ersst_ann["sst"]), c="k")
plt.plot(get_e(X_ersst_ann_filt["sst"]), c="r")
# plt.plot(fn(X_ersst_ann["sst"]), c="k")
# plt.plot(fn(X_ersst_ann["sst"]), c="k")
# plt.plot(fn(X_ersst_ann_recon))

In [ ]:
X_ersst_ann["year_"] = X_ersst_ann.year.values * xr.ones_like(X_ersst_ann["T0"])
X_ersst_ann_filt["year_"] = X_ersst_ann_filt.year.values * xr.ones_like(
    X_ersst_ann["T0"]
)
X_ersst_ann_filt["T0"] = X_ersst_ann["T0"]

trend = regress_pinv(
    X_ersst_ann, y_var="sst", x_vars=["year_", "T0"], dims=["year"]
).sel(param="year_")
trend_filt = regress_pinv(
    X_ersst_ann_filt, y_var="sst", x_vars=["year_", "T0"], dims=["year"]
).sel(param="year_")

trend = trend * 100
trend_filt = trend_filt * 100

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(trend.longitude, trend)
ax.plot(trend.longitude, trend_filt)
ax.plot(rect_ersst.longitude, -rect_ersst.mean("month") * 10)
ax.axhline(0, **ax_kwargs)
src.utils.add_vticks([ax], xticks=[80, 180, 280], xlines=[180])

Could do intermodel scatter: how much does gradient depend on ENSO?

Question: how much does ENSO project on zonal gradient...  
If it's diff. for models/obs there will be a difference...  
Could also regress variance spatial pattern onto gradient index...